# 15 - Final results and implementation log
Spec notebook 15 / plan F4-F5. Loads the saved records of notebooks 03-14 (and the Orin report once it exists), **checks their provenance** (full runs, same split hash, same checkpoints, every downstream run built on the attached upstream run), and builds from them only:
- the **metrics catalogue** (spec section 16: definition, unit, aggregation, split, hardware, source path, value, or `NOT MEASURED` with the reason),
- the **result tables** and **plots** (each plot labelled with model, data split, hardware and run ID),
- a draft **final implementation log** (spec section 39, 30 sections) as Markdown, plus `final_results.json`.

**No number is computed or estimated here.** A missing record shows as `MISSING`; Orin metrics are `NOT MEASURED` until the Jetson runbook has run. Narrative sections that no record can supply (repository audit, files inspected) are marked as such, to be completed by hand.

**Inputs (Add Input -> Your Work):** the full-run outputs of notebooks 03, 04, 05, 06, 07, 08, 09, 10, 11, 12, 13, 14 (Orin report later).
**Accelerator:** none (CPU). Internet ON (repo clone; the unit tests run in this notebook).
**How to run:** Run All (no smoke mode: nothing heavy runs). Then **Save Version -> Save & Run All**.

In [ ]:
# --- Setup: get the project code onto Kaggle ---
# Preferred: attach the code as a Kaggle dataset (works for a private GitHub repo).
# Fallback: git clone (only works if the GitHub repo is public and Internet is ON).
import os, sys, glob, shutil, subprocess
REPO_URL = "https://github.com/Amith2277/Edge-Computing-EndSem.git"
REPO_DIR = "/kaggle/working/repo"
MARK = os.path.join("src", "common", "config.py")

def find_attached_code():
    for pat in ("/kaggle/input/*/", "/kaggle/input/*/*/", "/kaggle/input/*/*/*/"):
        for d in sorted(glob.glob(pat)):
            if os.path.isfile(os.path.join(d, MARK)):
                return d.rstrip("/")
    return None

if os.path.isdir(REPO_DIR):
    print("using existing", REPO_DIR)
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):  # cloned earlier: fetch the latest code
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout.strip())
else:
    attached = find_attached_code()
    if attached:
        shutil.copytree(attached, REPO_DIR)
        print("copied attached code dataset:", attached)
    else:
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout)
        if r.returncode != 0:
            raise SystemExit("No attached code dataset found and git clone failed (private repo?). "
                             "Upload adaptivepark-code.zip as a Kaggle dataset and Add Input it, then re-run.")
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("code at", REPO_DIR, "| files:", sorted(os.listdir(REPO_DIR))[:12])


In [ ]:
# Settings, discovery, provenance checks
RUN_TESTS = True                  # run the unit tests of the cloned repo and record the result (log section 26)
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"
import os, json, subprocess, sys, csv
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
from src.common.config import load_yaml, config_hash
from src.common.status import new_run_id, write_status
from src.reporting import final as fr
FRAC = load_yaml("configs/dataset.yaml")["dataset"]["subset_fraction"]
FOUND = fr.discover(INPUT_ROOT, FRAC)
R = {k: v["record"] for k, v in FOUND.items()}
CHECKS = fr.checks(FOUND)
OUT = Path(WORK) / "artifacts" / "report"
(OUT / "plots").mkdir(parents=True, exist_ok=True)
RUN_ID = new_run_id("rep")
CFG_HASH = config_hash({k: (v["record"] or {}).get("run_id") for k, v in FOUND.items()})
STATUS = OUT / "status.json"
write_status(STATUS, "15_final_results_and_report", RUN_ID, "RUNNING", CFG_HASH, "records discovered")
print("{:<14} {:<16} {:<34} {}".format("record", "notebook", "run id", "file"))
for k, v in FOUND.items():
    print("{:<14} {:<16} {:<34} {}".format(k, v["spec"]["notebook"], str((v["record"] or {}).get("run_id", "-")), v["path"] or "NOT FOUND"))
bad = [c for c in CHECKS if c["result"] == "FAIL"]
print("\nprovenance checks: {} pass, {} fail, {} skip".format(sum(c["result"] == "PASS" for c in CHECKS), len(bad), sum(c["result"] == "SKIP" for c in CHECKS)))
for c in bad:
    print("  FAIL", c["check"], "|", c["detail"])

In [ ]:
# Metrics catalogue (spec section 16: every metric with definition, unit, source, or NOT MEASURED + reason)
CAT = fr.catalogue_rows(R)
def flat(r):
    return {**{k: r[k] for k in ("section", "metric", "definition", "unit", "aggregation", "split", "hardware", "source", "note")},
            **{"value_" + k: fr.fmt(v) for k, v in r["values"].items()}}
with open(str(OUT / "metrics_catalogue.csv"), "w", newline="", encoding="utf-8") as f:
    rows = [flat(r) for r in CAT]
    cols = sorted({k for r in rows for k in r}, key=lambda k: (not k.startswith(("section", "metric")), k))
    w = csv.DictWriter(f, fieldnames=["section", "metric", "definition", "unit", "aggregation", "split", "hardware", "source", "note",
                                      "value", "value_yolo11n", "value_yolo11s"], extrasaction="ignore")
    w.writeheader(); [w.writerow(r) for r in rows]
vals = [v for r in CAT for v in r["values"].values()]
count = lambda s: sum(v == s for v in vals)
print("catalogue: {} metrics, {} values | NOT MEASURED {} | MISSING {} | PLOT / TABLE references {}".format(
      len(CAT), len(vals), count("NOT MEASURED"), count(fr.MISSING), count("PLOT") + count("TABLE")))
miss = [(r["section"], r["metric"], k, r["source"]) for r in CAT for k, v in r["values"].items() if v == fr.MISSING]
for m in miss:
    print("  MISSING", m)
CAT_MD = fr.md_table([{"section": r["section"], "metric": r["metric"], "unit": r["unit"], "split": r["split"], "hardware": r["hardware"],
                       "yolo11n / value": r["values"].get("yolo11n", r["values"].get("value")), "yolo11s": r["values"].get("yolo11s", ""),
                       "source / reason": r["source"] or r["note"]} for r in CAT],
                     ["section", "metric", "unit", "split", "hardware", "yolo11n / value", "yolo11s", "source / reason"])

In [ ]:
# Result tables, read from the records only
g = fr.get
T = {}
def val(rec, path):
    x = g(rec, path); return None if x == fr.MISSING else x
lat = lambda v, dev, prec, b: val(R.get("bench"), "model_latency[model={},device={},precision={},batch={}]/per_image_ms/p50".format(v, dev, prec, b))
T["T1 model comparison (test split; latency Kaggle T4 / Xeon, NOT Jetson)"] = fr.md_table([{
    "model": v, "params": val(R.get("train:" + v), "parameters"), "GFLOPs": val(R.get("train:" + v), "gflops_at_imgsz"),
    "ckpt MB": None if val(R.get("train:" + v), "checkpoint_size_bytes") is None else round(val(R.get("train:" + v), "checkpoint_size_bytes") / 2 ** 20, 2),
    "imgsz": val(R.get("train:" + v), "imgsz"), "mAP50": val(R.get("det:" + v), "test/mAP50"), "mAP50-95": val(R.get("det:" + v), "test/mAP50-95"),
    "P": val(R.get("det:" + v), "test/at_operating_conf/precision"), "R": val(R.get("det:" + v), "test/at_operating_conf/recall"),
    "F1": val(R.get("det:" + v), "test/at_operating_conf/f1"), "occ macro-F1": val(R.get("occ:" + v), "test/macro_f1"),
    "GPU fp16 b1 ms": lat(v, "cuda:0", "fp16", 1), "GPU fp16 b16 ms/img": lat(v, "cuda:0", "fp16", 16), "CPU fp32 b1 ms": lat(v, "cpu", "fp32", 1),
    "peak GPU MiB": val(R.get("bench"), "model_latency[model={},device=cuda:0,precision=fp16,batch=1]/gpu_peak_mem_mib".format(v))} for v in fr.MODELS],
    ["model", "params", "GFLOPs", "ckpt MB", "imgsz", "mAP50", "mAP50-95", "P", "R", "F1", "occ macro-F1", "GPU fp16 b1 ms", "GPU fp16 b16 ms/img", "CPU fp32 b1 ms", "peak GPU MiB"])
T["T2 occupancy, notebook 07 score threshold (test)"] = fr.md_table([dict(model=v, **(val(R.get("occ:" + v), "test") or {}),
    mae=val(R.get("occ:" + v), "test_free_slot_count_error/mae"), unknown=val(R.get("occ:" + v), "test_unknown_slots")) for v in fr.MODELS],
    ["model", "slots", "accuracy", "macro_f1", "balanced_accuracy", "FV", "FO", "false_vacant_rate", "false_occupied_rate", "mae", "unknown"])
rows = []
for v in fr.MODELS:
    for grp, s_ in sorted((val(R.get("occ:" + v), "groups") or {}).items()):
        if grp.startswith(("site:", "climate:", "camera:")):
            rows.append({**s_, "model": v, "group": grp})
T["T3 occupancy per site / climate / camera (test, notebook 07)"] = fr.md_table(rows, ["model", "group", "slots", "accuracy", "macro_f1", "FV", "FO", "note"])
rows = []
for v in fr.MODELS:
    for which in ("test_none", "test_chosen"):
        x = val(R.get("tmp:" + v), which)
        if x:
            rows.append({**x, "model": v})
T["T4 temporal validation (test, notebook 08 score threshold)"] = fr.md_table(rows, ["model", "setting", "accuracy", "macro_f1", "FV", "FO", "detection_rate",
    "false_per_1000_slot_frames", "pred_transitions_per_slot_hour", "mean_delay_frames", "mean_delay_minutes"])
rows = []
for v in fr.MODELS:
    for ratio, ch in (val(R.get("cost"), "decision_rule_per_ratio/" + v) or {}).items():
        base = val(R.get("cost"), "test_rows[model={},ratio_fv_fo={},rule=nb07 score threshold]".format(v, ratio))
        chosen = val(R.get("cost"), "test_rows[model={},ratio_fv_fo={},rule={}]".format(v, ratio, ch["rule"]))
        rows.append({"model": v, "C_FV/C_FO": ratio, "val-chosen rule": ch["rule"], "threshold": ch.get("threshold"),
                     "test cost nb07": (base or {}).get("cost_per_1000"), "test cost chosen": (chosen or {}).get("cost_per_1000"),
                     "FV nb07": (base or {}).get("FV"), "FV chosen": (chosen or {}).get("FV"), "FO chosen": (chosen or {}).get("FO")})
T["T5 cost-sensitive decision per cost ratio (test; cost per 1000 slot decisions, C_FO = 1)"] = fr.md_table(rows, list(rows[0]) if rows else [])
rows = []
for v in fr.MODELS:
    q = val(R.get("cost"), "calibration/{}/quality".format(v)) or {}
    for m_, x in q.items():
        rows.append({"model": v, "calibrator": m_, "chosen": val(R.get("cost"), "calibration/{}/chosen_by_val_log_loss".format(v)) == m_,
                     **{"test " + k: x.get("test", {}).get(k) for k in ("brier", "log_loss", "ece")}})
T["T5b calibration (test; chosen by val cross-fitted log loss)"] = fr.md_table(rows, ["model", "calibrator", "chosen", "test brier", "test log_loss", "test ece"])
res = val(R.get("eval"), "results/test") or {}
T["T6 end-to-end strategies, both decision rules (test, notebook 13; compute = Kaggle T4 batched ratio)"] = fr.md_table(
    [{**x, "name": k} for k, x in res.items()], ["name", "accuracy", "macro_f1", "FV", "FO", "cost_per_1000_decisions", "cost_per_frame", "skipped_share",
    "stale_share", "false_flips_per_1000", "transition_delay_frames", "max_age_violations"])
rows = []
for site, d_ in (val(R.get("eval"), "sites_test_deployed") or {}).items():
    for k, x in d_.items():
        rows.append({**x, "site": site, "strategy": k})
T["T6b per site, deployed rule (test, notebook 13)"] = fr.md_table(rows, ["site", "strategy", "accuracy", "macro_f1", "cost_per_1000_decisions", "compute_per_frame"])
T["T7 ablations of AdaptivePark, deployed rule (test, notebook 13)"] = fr.md_table(
    [{**x, "name": k} for k, x in (val(R.get("eval"), "ablations_test") or {}).items()], ["name", "accuracy", "macro_f1", "FV", "FO", "cost_per_1000_decisions", "cost_per_frame", "false_flips_per_1000"])
rows = []
for k, d_ in (val(R.get("eval"), "bootstrap_test") or {}).items():
    for s_, x in d_.items():
        rows.append({"comparison (A - B)": k, "statistic": s_, "difference": x["difference"], "ci95 low": x["ci95"][0], "ci95 high": x["ci95"][1],
                     "share > 0": x["share_above_0"], "sequences": x["sequences"]})
T["T8 paired bootstrap over camera-day sequences (test, notebook 13)"] = fr.md_table(rows, list(rows[0]) if rows else [])
dt = val(R.get("drift"), "test_chosen_on_val")
T["T9 drift monitor, synthetic drift (test, notebook 12, test chosen on val: {})".format(dt)] = fr.md_table(
    val(R.get("drift"), "synthetic/test/{}".format(dt)) or [], ["variant", "sequences", "alert_rate_drifted", "critical_rate_drifted", "alert_rate_before_onset",
    "mean_first_alert_delay_frames", "accuracy_drop"])
T["T9b drift thresholds: clean val windows per test (notebook 12)"] = fr.md_table(
    [dict(test=k, saturated=x.get("saturated"), **{"clean " + s_: r_ for s_, r_ in (x.get("clean_rate") or {}).items()}) for k, x in (val(R.get("drift"), "thresholds") or {}).items()],
    ["test", "clean ok", "clean warning", "clean critical", "saturated"])
T["T10 Kaggle model latency, PyTorch ({})".format(val(R.get("bench"), "label"))] = fr.md_table(
    [dict(model=x["model"], device=x["device"], precision=x["precision"], batch=x["batch"], p50=x["per_image_ms"]["p50"], p95=x["per_image_ms"]["p95"],
          p99=x["per_image_ms"]["p99"], mean=x["per_image_ms"]["mean"], fps=x["fps_images"], mem=x.get("gpu_peak_mem_mib")) for x in (val(R.get("bench"), "model_latency") or [])],
    ["model", "device", "precision", "batch", "p50", "p95", "p99", "mean", "fps", "mem"], ["model", "device", "precision", "batch", "p50 ms/img", "p95", "p99", "mean", "img/s", "peak GPU MiB"])
T["T10b Kaggle end-to-end FramePipeline per frame (NOT Jetson)"] = fr.md_table(
    [dict(name=k, frames=x["frames"], p50=x["total_ms"]["p50"], p95=x["total_ms"]["p95"], mean=x["total_ms"]["mean"], fps=x["fps"], accuracy=x["accuracy"],
          **{s_: m_ for s_, m_ in x["stage_mean_ms"].items()}) for k, x in (val(R.get("bench"), "pipeline") or {}).items()],
    ["name", "frames", "p50", "p95", "mean", "fps", "accuracy", "decode", "scene", "decide", "detect", "associate", "update", "drift"])
for k, t in T.items():
    print("### " + k); print(t)

In [ ]:
# Plots from the records (each labelled with data split, hardware and run ID)
import matplotlib.pyplot as plt
PLOTS = []
def save(fig, name):
    p = OUT / "plots" / name; fig.tight_layout(); fig.savefig(str(p), dpi=110); plt.show(); PLOTS.append(str(p))
if res:
    fig, a = plt.subplots(figsize=(8, 5))
    for k, x in res.items():
        a.scatter(x["cost_per_frame"], x["cost_per_1000_decisions"], marker="o" if "deployed" in k else "x")
        a.annotate(k.replace(" | ", "\n"), (x["cost_per_frame"], x["cost_per_1000_decisions"]), fontsize=6, xytext=(3, 3), textcoords="offset points")
    a.set(xlabel="compute per frame (YOLO11n units; Kaggle T4 batched ratio)", ylabel="cost per 1000 slot decisions (C_FV/C_FO = 5)",
          title="Strategies, test split, replay of cached n/s detections\nnotebook 13 run {}".format(val(R.get("eval"), "run_id"))); a.grid(alpha=0.3)
    save(fig, "strategies_cost_vs_compute.png")
if R.get("cost"):
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5), sharey=False)
    for a, v in zip(ax, fr.MODELS):
        for rule in sorted({r["rule"] for r in R["cost"]["test_rows"] if r["model"] == v}):
            rr = sorted([r for r in R["cost"]["test_rows"] if r["model"] == v and r["rule"] == rule], key=lambda r: r["ratio_fv_fo"])
            a.plot([r["ratio_fv_fo"] for r in rr], [r["cost_per_1000"] for r in rr], marker="o", label=rule)
        a.set(xscale="log", xlabel="C_FV / C_FO", ylabel="test cost per 1000 decisions", title="{} | test | notebook 11 run {}".format(v, R["cost"]["run_id"]))
        a.grid(alpha=0.3); a.legend(fontsize=7)
    save(fig, "cost_vs_ratio.png")
if R.get("drift"):
    fig, a = plt.subplots(figsize=(8, 5))
    for x in val(R["drift"], "synthetic/test/" + dt) or []:
        a.scatter(x["accuracy_drop"], x["alert_rate_drifted"]); a.annotate(x["variant"], (x["accuracy_drop"], x["alert_rate_drifted"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
    a.set(xlabel="slot accuracy drop after onset (YOLO11n re-detection)", ylabel="alert rate of fully drifted windows",
          title="Synthetic drift, test split, {} on 8-frame windows | notebook 12 run {}".format(dt, R["drift"]["run_id"])); a.grid(alpha=0.3)
    save(fig, "drift_alert_vs_accuracy_drop.png")
if R.get("bench"):
    L = R["bench"]["model_latency"]
    fig, a = plt.subplots(figsize=(9, 4.5))
    lab = ["{} {} {} b{}".format(x["model"], x["device"].split(":")[0], x["precision"], x["batch"]) for x in L]
    a.barh(range(len(L)), [x["per_image_ms"]["p50"] for x in L]); a.set_yticks(range(len(L))); a.set_yticklabels(lab, fontsize=8); a.set_xscale("log")
    a.set(xlabel="median ms per image (PyTorch)", title="{}\nnotebook 14 run {}".format(R["bench"]["label"], R["bench"]["run_id"])); a.grid(alpha=0.3, axis="x")
    save(fig, "kaggle_latency.png")
EXISTING = sorted(str(p) for p in Path(INPUT_ROOT).rglob("*.png") if "artifacts" in str(p))
print("plots made here:", PLOTS); print("plots saved by notebooks 03-14 ({}):".format(len(EXISTING)), EXISTING[:60])

In [ ]:
# Repository state: commit, unit tests, which parts exist (API, dashboard, Orin scripts)
def sh(cmd, timeout=900):
    try:
        r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True, timeout=timeout)
        return r.returncode, r.stdout
    except Exception as e:
        return -1, repr(e)
REPO = Path.cwd()
COMMIT = sh(["git", "-C", str(REPO), "log", "-1", "--format=%H %cI %s"])[1].strip()
GITLOG = sh(["git", "-C", str(REPO), "log", "--format=%h %cs %s", "-n", "40"])[1].strip()
FILES = sh(["git", "-C", str(REPO), "ls-files"])[1].split()
by_dir = defaultdict(int)
for f_ in FILES:
    by_dir[f_.split("/")[0] if "/" in f_ else "(root)"] += 1
TESTS = {"run": False}
if RUN_TESTS:
    rc, outp = sh([sys.executable, "-m", "pytest", "-q", "tests/unit"])
    TESTS = {"run": True, "returncode": rc, "summary": (outp.strip().splitlines() or ["(no output)"])[-1]}
REPO_HAS = {"FastAPI backend (plan H1)": any(f_.startswith(("src/api/", "app/api/")) for f_ in FILES),
            "dashboard (plan H2)": any(f_.startswith(("dashboard/", "src/dashboard/", "app/dashboard/")) for f_ in FILES),
            "Orin scripts (plan G1-G8)": all(any(f_.endswith(n_) for f_ in FILES) for n_ in ("check_jetson_environment.py", "export_onnx.py", "build_tensorrt.py", "run_replay.py", "benchmark_jetson.py"))}
STATE = fr.overall_status(FOUND, REPO_HAS, CHECKS)
print("commit:", COMMIT); print("files by top-level folder:", dict(by_dir)); print("unit tests:", TESTS); print("repo has:", REPO_HAS); print("status:", STATE)

In [ ]:
# Final implementation log (spec section 39) + final_results.json
def sec(n, title, body):
    return "## {}. {}\n\n{}\n".format(n, title, body.strip() + "\n")
HAND = "_Not derivable from saved records: to be completed by hand from the project plan._"
def kv(d):
    return fr.md_table([{"field": k, "value": x} for k, x in d.items()], ["field", "value"]) if d else "_(not available)_\n"
def nb_status():
    rows = []
    for k, v in FOUND.items():
        r_ = v["record"] or {}
        rows.append({"record": k, "notebook": v["spec"]["notebook"], "found": v["record"] is not None, "run_id": r_.get("run_id"),
                     "full run": (not r_.get("smoke_test")) if v["record"] is not None and "smoke_test" in r_ else None,
                     "status": (v["status"] or {}).get("state"), "file": v["path"]})
    return fr.md_table(rows, ["record", "notebook", "found", "run_id", "full run", "status", "file"])
def summary():
    b = []
    for v in fr.MODELS:
        b.append("- {}: test mAP50-95 {}, occupancy accuracy {} (notebook 07 rule), cost@5 {} per 1000 with the deployed rule (fixed model, notebook 13).".format(
            v, fr.fmt(val(R.get("det:" + v), "test/mAP50-95")), fr.fmt(val(R.get("occ:" + v), "test/accuracy")),
            fr.fmt(val(R.get("eval"), "results/test/fixed {} | deployed/cost_per_1000_decisions".format(v.replace("yolo", "YOLO"))))))
    ap = val(R.get("eval"), "results/test/AdaptivePark (accuracy point) | deployed") or {}
    b.append("- AdaptivePark (deployed rule): macro-F1 {}, cost@5 {} per 1000, compute {} YOLO11n units per frame (Kaggle T4 batched ratio).".format(
        fr.fmt(ap.get("macro_f1")), fr.fmt(ap.get("cost_per_1000_decisions")), fr.fmt(ap.get("cost_per_frame"))))
    for k in ("AdaptivePark vs fixed YOLO11n (deployed rule)", "AdaptivePark vs fixed YOLO11s (deployed rule)"):
        x = val(R.get("eval"), "bootstrap_test/{}/cost@5 per 1000".format(k))
        if x:
            b.append("- {}: cost@5 difference {} [95% {} .. {}].".format(k, fr.fmt(x["difference"]), fr.fmt(x["ci95"][0]), fr.fmt(x["ci95"][1])))
    b.append("- Kaggle batch-1 YOLO11s / YOLO11n latency ratio: {} (NOT Jetson).".format(fr.fmt(val(R.get("bench"), "latency_ratio_s_over_n"))))
    b.append("- Jetson AGX Orin: {}.".format("report attached" if R.get("jetson") else "NOT MEASURED (" + fr.NO_JETSON + ")"))
    return "\n".join(b)
LIMITS = "\n".join("- **{}**: {}".format(k, r_["limitation"]) for k, r_ in R.items() if r_ and r_.get("limitation"))
log = ["# AdaptivePark - final implementation log (generated draft)\n",
       "Generated by notebook 15 run `{}` at {} from commit `{}`. Every number below is read from a saved record; nothing is estimated.\n".format(
           RUN_ID, datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC"), COMMIT)]
log.append(sec(1, "Status", "**{}**\n\n- missing records: {}\n- failed provenance checks: {}\n- not done: {}".format(
    STATE["state"], STATE["missing_records"] or "none", STATE["failed_checks"] or "none", STATE["gaps"] or "none")))
log.append(sec(2, "Executive summary", summary()))
log.append(sec(3, "Original repository audit", HAND))
log.append(sec(4, "Files inspected", HAND))
log.append(sec(5, "Files created, modified or deleted", "Tracked files by folder at this commit:\n\n" + kv(dict(by_dir)) + "\nLast commits:\n\n```\n" + GITLOG + "\n```"))
log.append(sec(6, "Notebook-by-notebook status", nb_status()))
sp_ = R.get("split") or {}
scal = {k: x for k, x in sp_.items() if not isinstance(x, (dict, list))}
log.append(sec(7, "Dataset source, version and schema", kv(scal)))
log.append(sec(8, "Dataset counts and annotation availability", kv({k: x for k, x in sp_.items() if isinstance(x, dict)}) + "\nTest images / cars: {} / {} (notebook 06).".format(
    fr.fmt(val(R.get("det:yolo11n"), "test/images")), fr.fmt(val(R.get("det:yolo11n"), "test/cars")))))
log.append(sec(9, "Split strategy and hashes", "Manifest hash `{}`; every downstream record is checked against it (section 6 and the provenance table).".format(sp_.get("manifest_hash"))))
log.append(sec(10, "Model training versus pretrained-only status", "\n".join("- {}: {}".format(v, val(R.get("train:" + v), "status")) for v in fr.MODELS)))
log.append(sec(11, "Checkpoint paths and hashes", fr.md_table([{"model": v, "sha256": val(R.get("train:" + v), "checkpoint_sha256"),
    "bytes": val(R.get("train:" + v), "checkpoint_size_bytes"), "run": val(R.get("train:" + v), "run_id")} for v in fr.MODELS], ["model", "sha256", "bytes", "run"])))
log.append(sec(12, "Best epochs and training settings", fr.md_table([{"model": v, **{k: val(R.get("train:" + v), k) for k in ("epochs_configured", "epochs_completed",
    "imgsz", "batch", "optimizer_used", "seed", "training_precision", "training_duration_s")}, "best epoch": val(R.get("train:" + v), "best_epoch/epoch")} for v in fr.MODELS],
    ["model", "epochs_configured", "epochs_completed", "best epoch", "imgsz", "batch", "optimizer_used", "seed", "training_precision", "training_duration_s"])))
log.append(sec(13, "D / R formulas and selected weights", "D = sum_i w_i F_i over B, M, C, V, O, S (missing features re-normalised); R = alpha U + beta Delta + gamma A + delta O.\n\n"
    "D weights: {}\n\nscales: {}\n\nR weights: {}\n\nval objective: {} / {}".format(fr.fmt(val(R.get("scene"), "difficulty/weights")),
    fr.fmt(val(R.get("scene"), "difficulty/scales")), fr.fmt(val(R.get("scene"), "slot_risk/weights")), val(R.get("scene"), "difficulty/objective"), val(R.get("scene"), "slot_risk/objective"))))
log.append(sec(14, "Thresholds, cadence, hysteresis and dwell time", "Accuracy point (deployed):\n\n" + kv(val(R.get("ctl"), "chosen/adaptive/accuracy_point") or {})
    + "\nEfficiency point:\n\n" + kv(val(R.get("ctl"), "chosen/adaptive/efficiency_point") or {}) + "\nSelection rule: {}".format(val(R.get("ctl"), "selection_rule"))))
log.append(sec(15, "Risk aggregation method", fr.fmt(val(R.get("scene"), "slot_risk/aggregation"))))
log.append(sec(16, "Maximum observation age", "{} s (accuracy point); test max-age violations {} (notebook 13, deployed rule).".format(
    fr.fmt(val(R.get("ctl"), "chosen/adaptive/accuracy_point/max_age_s")), fr.fmt(val(R.get("eval"), "results/test/AdaptivePark (accuracy point) | deployed/max_age_violations")))))
log.append(sec(17, "Cost-sensitive configuration", kv(val(R.get("eval"), "decision/deployed") or {})))
log.append(sec(18, "Temporal validation settings", "\n".join("- {} (notebook 08 macro-F1 choice): {}".format(v, val(R.get("tmp:" + v), "chosen_on_val/setting")) for v in fr.MODELS)
    + "\n- deployed (notebook 11 cost choice): {}".format(fr.fmt(val(R.get("eval"), "decision/deployed/filter")))))
log.append(sec(19, "Baselines and ablations executed", "Strategies: {}\n\nAblations: {}\n\nRule: {}".format(", ".join(sorted({k.split(" | ")[0] for k in res})),
    ", ".join(val(R.get("eval"), "ablations_test") or {}), val(R.get("eval"), "ablation_rule"))))
log.append(sec(20, "Metric tables (loaded from saved result files)", "\n".join("### {}\n\n{}".format(k, t) for k, t in T.items()) + "\n### Metrics catalogue\n\n" + CAT_MD))
log.append(sec(21, "Plot paths", "Made by this notebook:\n\n" + "\n".join("- " + p for p in PLOTS) + "\n\nSaved by notebooks 03-14:\n\n" + "\n".join("- " + p for p in EXISTING)))
log.append(sec(22, "Kaggle hardware and results", kv(val(R.get("bench"), "hardware") or {}) + "\nLabel: {}. Tables T1, T10, T10b.".format(val(R.get("bench"), "label"))))
log.append(sec(23, "Jetson hardware / software identity", kv(val(R.get("jetson"), "environment") or {}) if R.get("jetson") else "NOT MEASURED: " + fr.NO_JETSON))
log.append(sec(24, "Jetson measurements", "See section 22 of the metrics catalogue." if R.get("jetson") else "NOT MEASURED: " + fr.NO_JETSON + ". Every section-22 metric is listed as NOT MEASURED in the catalogue."))
log.append(sec(25, "ONNX / TensorRT validation status", "ONNX (Kaggle rehearsal, notebook 14):\n\n" + fr.md_table(
    [dict(model=v, **{k: x for k, x in (val(R.get("bench"), "onnx/" + v) or {}).items() if not isinstance(x, dict)}) for v in fr.MODELS],
    ["model", "status", "opset", "size_mb", "box_match_share_mean", "box_match_share_at_operating_threshold", "conf_abs_diff_max", "reason"])
    + "\nTensorRT: NOT RUN (engines are built on the Orin, plan G5)."))
log.append(sec(26, "Test commands and results", "`python -m pytest -q tests/unit` in this notebook: {}".format(TESTS.get("summary", "not run"))))
log.append(sec(27, "Reproduction instructions", "Run on Kaggle in order, each with the previous outputs attached (Add Input -> Your Work), SMOKE = True first, then SMOKE = False + Save & Run All: "
    "03 splits, 04 / 05 training (GPU), 06 detection evaluation (GPU), 07 occupancy (GPU), 08 temporal, 09 scene / risk, 10 controller, 11 cost-sensitive, "
    "12 drift (GPU), 13 end-to-end evaluation, 14 Kaggle benchmark (GPU), 15 this report. Notebooks are generated from the repo; the repo is cloned at the top of each."))
log.append(sec(28, "Blockers and deviations", "\n".join(["- failed check: {} ({})".format(c["check"], c["detail"]) for c in CHECKS if c["result"] == "FAIL"] + ["- not done: " + x for x in STATE["gaps"]]) or "none"))
log.append(sec(29, "Limitations", LIMITS or "_(none recorded)_"))
log.append(sec(30, "Evidence-based conclusion and next actions", "Status {}. Next actions: {}.".format(STATE["state"], "; ".join(STATE["gaps"]) or "none")))
LOG = "\n".join(log)
(OUT / "final_implementation_log.md").write_text(LOG, encoding="utf-8")
FINAL = {"run_id": RUN_ID, "config_hash": CFG_HASH, "commit": COMMIT, "status": STATE, "checks": CHECKS, "tests": TESTS, "repo_has": REPO_HAS,
         "records": {k: {"notebook": v["spec"]["notebook"], "path": v["path"], "run_id": (v["record"] or {}).get("run_id")} for k, v in FOUND.items()},
         "catalogue": [{k: x for k, x in r.items()} for r in CAT], "plots": PLOTS}
(OUT / "final_results.json").write_text(json.dumps(FINAL, indent=1, default=str))
write_status(STATUS, "15_final_results_and_report", RUN_ID, "COMPLETE", CFG_HASH, STATE["state"], extra={"failed_checks": len(STATE["failed_checks"]), "missing": STATE["missing_records"]})
print(LOG[:4000]); print("...\nCOMPLETE | saved to", OUT, sorted(p.name for p in OUT.iterdir()))

## What to send back
Download the notebook after **Save & Run All** and send it. `artifacts/report/final_implementation_log.md` is the draft of `docs/final_implementation_log.md`; `final_results.json` and `metrics_catalogue.csv` hold every value with its source.